In [1]:
#read needed packages, should execute everytime
#version 0.2 (newer than alpha, which includes some pre-processing steps)
import pandas as pd
import time
import csv
import re
import ast

In [2]:
#read glycotope settings if there is one
#in alpha version it's hand-written in below cell
try:
    theoglycotopedict = pd.read_csv("glycotope_deducer.csv")
except:
    theoglycotopedict = {}

In [3]:
#hand writing rules of glycotopes
glycotopeheader = ["Name of glycotope feature", "used units", "enzyme list and presence 0,1 or ?", "ion list", "groups"]
#explanations: name: name, used units: no. of units used for calculation, if we can deduce from structure, not from the
#monosaccharide units, then this part will become depreciated. enzyme list: 0 for absence, 1 for presence, "?" for unknown
#ion list = list of ions supporting this epitope, 
#groups: stem = N-glycan core, only used when other starting cores are not found, core: N-glycan cores (1 exclusive for each search)
#ext: extension possible, ter: terminal epitope, no longer extensible.
#type1: based on type1 lacnac, type2: based on type2 lacnac


#this is defining epitope part
#Maybe it's better to keep values in dict as the same datatype
NG_core= ["NG core", {"H":3, "N":2}, {"MGAT1":1 ,"MGAT2":1, "MAN2A1":1 ,"MAN2A2":1, "unknown enzyme": '?'}, [1234.56, 888.88], "stem"]
Bisecting_NG_core= ["Bisecting NG core", {"H":3, "N":3}, {"MGAT3": 1}, [1234.56, 888.88], "core"]
Triante_NG_core= ["Tri-antennary N-glycan core", {"H":6, "N":5}, {"MGAT4":1 ,"MGAT5":1}, [1234.56, 888.88], "core"]
NG_elongationGal = ["First and elongation of Galactose at NG ", {"H":1}, {"B4GALT1":1, "B4TALT2~7": 1}, [187.0965, 209.1227], "ext"] #re?
corefuc = ["N-glycan core Fucosylation", {"F":1}, {"FUT8":1}, [452.2490, 697.3753], "ter"]
LacNAc1 = ["Type 1 LacNAc", {"H":1, "N":1}, {"B3GALT2":1, "B3GALT5":1}, [464], "ext, type1"]
LacNAc2 = ["Type 2 LacNAc", {"H":1, "N":1}, {"B4GALT1~7":1}, [464, 432], "ext, type2"]
H_type1 = ["a2Fuc-Type1 LNAc", {"F":1, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ter, type1"]
H_type2 = ["a2Fuc-Type2 LNAc", {"F":1, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ter, type2"]
LeA = ["Lewis A antigen", {"F":1, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ext, ter, type1"]#b3-Gal type1 w a4-Fuc
LeX = ["Lewis X antigen", {"F":1, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ext, ter, type2"]#b4-Gal type2 w a3-Fuc
LeB = ["Lewis B antigen", {"F":2, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ter, type1"]#b3-Gal type1 w a4-Fuc, a2-Fuc at Gal
LeY = ["Lewis Y antigen", {"F":2, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ter, type2"]#b4-Gal type2 w a3-Fuc, a2-Fuc at Gal
Neu5Ac = ["Neu5Ac sialic acid", {"S": 1}, {"CMAS": 1}, [376],  "ext, ter"]
Neu5Gc = ["Neu5Gc sialic acid", {"G": 1}, {"CMAH": 1}, [406], "ext, ter"]
KDN = ["KDN sialic acid", {"KDN": 1}, {"CMP-KDN synthetase": 1}, [335], "ter"]
Sia3_LNAc1 = ["a2-3-Sialylated Type 1 LNAc", {"H":1, "N":1, "S":1}, {}, [376, 580, 792, 825], "ext, type1"]
Sia6_LNAc1 = ["a2-6-Sialylated Type 1 LNAc", {"H":1, "N":1, "S":1}, {}, [376, 580, 792, 825], "ext, type1"]
Sia3_LNAc2 = ["a2-3-Sialylated Type 2 LNAc", {"H":1, "N":1, "S":1}, {}, [376, 580, 792, 825], "ext, type2"]
Sia6_LNAc2 = ["a2-6-Sialylated Type 2 LNAc", {"H":1, "N":1, "S":1}, {}, [376, 580, 792, 825], "ext, type2"]
zf_GalExtension = ["a1-4 Galx2 + GlcNAc", {"H":2, "N":1}, {"unknown": "?"}, [668], "ext"]
zf_specific_epitope_5Ac = ["a1-4 Galx2 + GlcNAc + a2-3Sia + a1-3Fuc", {"F":1, "H":2, "N":1, "S":1}, {"unknown": "?"}, [1203], "ext"]
zf_specific_epitope_5Gc = ["a1-4 Galx2 + GlcNAc + a2-3Sia-Gc + a1-3Fuc", {"F":1, "H":2, "N":1, "G":1}, {"unknown": "?"}, [1233], "ext"]
LacDiNAc = ["LacDiNAc", {"N":2}, {}, [505], "ext, ter"]
Sia6_LacDiNAc = ["a2-6Sia_LacDiNAc", {"N":2}, {}, [505], "ext, ter"]



def calcmassinepitope():
    print("developing")
#thinking if we can split epitopes into single component
#thinking if we can calculate fragments back from predicted structure (does that make sense in this pipeline?)
#should the ion be unique in all groups to avoid counted multiple times?


glycoenzymelist = [
    glycotopeheader,
    NG_core,
    Bisecting_NG_core,
    NG_elongationGal, 
    corefuc,
    LacNAc1,
    LacNAc2]

tmp = []
for glycoepitopes in glycoenzymelist:
    tmp.append(glycoepitopes)
print(f"tmp is+ {tmp}")
c = pd.DataFrame(tmp)
c.columns = c.iloc[0]
c = c[1:]
#set first row as index column, copied from https://www.statology.org/pandas-set-first-row-as-header/
c


tmp is+ [['Name of glycotope feature', 'used units', 'enzyme list and presence 0,1 or ?', 'ion list', 'groups'], ['NG core', {'H': 3, 'N': 2}, {'MGAT1': 1, 'MGAT2': 1, 'MAN2A1': 1, 'MAN2A2': 1, 'unknown enzyme': '?'}, [1234.56, 888.88], 'stem'], ['Bisecting NG core', {'H': 3, 'N': 3}, {'MGAT3': 1}, [1234.56, 888.88], 'core'], ['First and elongation of Galactose at NG ', {'H': 1}, {'B4GALT1': 1, 'B4TALT2~7': 1}, [187.0965, 209.1227], 'ext'], ['N-glycan core Fucosylation', {'F': 1}, {'FUT8': 1}, [452.249, 697.3753], 'ter'], ['Type 1 LacNAc', {'H': 1, 'N': 1}, {'B3GALT2': 1, 'B3GALT5': 1}, [464], 'ext, type1'], ['Type 2 LacNAc', {'H': 1, 'N': 1}, {'B4GALT1~7': 1}, [464, 432], 'ext, type2']]


,Name of glycotope feature,used units,"enzyme list and presence 0,1 or ?",ion list,groups
1,NG core,"{'H': 3, 'N': 2}","{'MGAT1': 1, 'MGAT2': 1, 'MAN2A1': 1, 'MAN2A2'...","[1234.56, 888.88]",stem
2,Bisecting NG core,"{'H': 3, 'N': 3}",{'MGAT3': 1},"[1234.56, 888.88]",core
3,First and elongation of Galactose at NG,{'H': 1},"{'B4GALT1': 1, 'B4TALT2~7': 1}","[187.0965, 209.1227]",ext
4,N-glycan core Fucosylation,{'F': 1},{'FUT8': 1},"[452.249, 697.3753]",ter
5,Type 1 LacNAc,"{'H': 1, 'N': 1}","{'B3GALT2': 1, 'B3GALT5': 1}",[464],"ext, type1"
6,Type 2 LacNAc,"{'H': 1, 'N': 1}",{'B4GALT1~7': 1},"[464, 432]","ext, type2"


In [4]:
#this is defining species part (only runs once for each analysis)

zebrafish_glycoT_brain = {"MGAT1":1 ,"MGAT2":1, "MAN2A1":1 ,"MAN2A2":1, "absentenzyme": 0, "unknown": "?"}

#to get possible epitope (only runs once for each analysis)
def findepitopefrom_glycoT(species):
    expression_list = []
    for key, value in species.items():
        if value == 1:
            expression_list.append(key)
    return expression_list
#those not expressed glycoT and unknown glycoT isn't returned in this version, consider if we need a way to recycle and use them

zfdemo = findepitopefrom_glycoT(zebrafish_glycoT_brain)
print(zfdemo)

['MGAT1', 'MGAT2', 'MAN2A1', 'MAN2A2']


In [5]:
c["enzyme list and presence 0,1 or ?"][1]
#print(c.loc[c["enzyme list and presence 0,1 or ?"]).isin(zfdemo)])

{'MGAT1': 1, 'MGAT2': 1, 'MAN2A1': 1, 'MAN2A2': 1, 'unknown enzyme': '?'}

In [6]:
#to get current structure list

#useless since calculation only runs in dictionary...

structure_dict = {"F":1, "H":5 , "N": 4, "S":1}
def getdeducestructure(structure_dict):
    if structure_dict is None:
        struc_tmp = {}
    try:
        Fuc = struc_tmp["F"]
    except KeyError: #keyerror -> Nameerror -> Syntaxerror (traceback)
        Fuc = 0
    try:
        Hex = struc_tmp["H"]
    except KeyError:
        Hex = 0
    try:
        HexNAc = struc_tmp["N"]
    except KeyError:
        HexNAc = 0
    try:
        Neu5Ac = struc_tmp["S"]
    except KeyError:
        Neu5Ac = 0
    try:
        Neu5Gc = struc_tmp["G"]
    except KeyError:
        Neu5Gc = 0
    try:
        KDN = struc_tmp["KDN"]
    except KeyError:
        KDN = 0
    try:
        deduceunitlist = [Fuc, Hex, HexNAc, Neu5Ac, Neu5Gc, KDN]
        #calc = struc_tmp - NG_core[1]
        #need to calculate key value inside the dict, direct add/sub is not valid
    except:
        print("error")
        deduceunitlist = None
    return deduceunitlist

def deducestructure(composition, dealt_epitope):
    if composition is None:
        struc_tmp = [0, 0, 0, 0, 0, 0]
    else:
        struc_tmp = structure_dict
    try:
        print("try to deduce")
        #pseudocode:
        #for 
        #calc = struc_tmp - NG_core[1]
        #need to calculate key value inside the dict, direct add/sub is not valid
    except:
        print("error")
        calc = None
    return calc

#deducestructure(composition, structure_dict)

In [7]:
print(f"This is testing cell")
print(len(c.index))
for i in range(1,len(c.index)+1):
    print(c["enzyme list and presence 0,1 or ?"][i])

This is testing cell
6
{'MGAT1': 1, 'MGAT2': 1, 'MAN2A1': 1, 'MAN2A2': 1, 'unknown enzyme': '?'}
{'MGAT3': 1}
{'B4GALT1': 1, 'B4TALT2~7': 1}
{'FUT8': 1}
{'B3GALT2': 1, 'B3GALT5': 1}
{'B4GALT1~7': 1}


In [8]:
c.loc[1]

0
Name of glycotope feature                                                      NG core
used units                                                            {'H': 3, 'N': 2}
enzyme list and presence 0,1 or ?    {'MGAT1': 1, 'MGAT2': 1, 'MAN2A1': 1, 'MAN2A2'...
ion list                                                             [1234.56, 888.88]
groups                                                                            stem
Name: 1, dtype: object

In [9]:
#c.loc[c["enzyme list and presence 0,1 or ?"].isin(zfdemo)] is not applicable sice we're wrapping dict+tuples inside the column 
newdfl = []
zzz = 1
for i in c["enzyme list and presence 0,1 or ?"]:
    e =[]
    for key, value in i.items():
        if value == 1:
            e.append(key)
    s = list(set(e) & set(zfdemo))
    if s != []:
        newdfl.append(c.loc[zzz])
        #should be appending the whole row
    else:
        tmpoutput = i
        print(f"There is no enzyme {tmpoutput} for specified epitope")
    zzz +=1
    
newdf = pd.DataFrame(newdfl)

newdf

There is no enzyme {'MGAT3': 1} for specified epitope
There is no enzyme {'B4GALT1': 1, 'B4TALT2~7': 1} for specified epitope
There is no enzyme {'FUT8': 1} for specified epitope
There is no enzyme {'B3GALT2': 1, 'B3GALT5': 1} for specified epitope
There is no enzyme {'B4GALT1~7': 1} for specified epitope


,Name of glycotope feature,used units,"enzyme list and presence 0,1 or ?",ion list,groups
1,NG core,"{'H': 3, 'N': 2}","{'MGAT1': 1, 'MGAT2': 1, 'MAN2A1': 1, 'MAN2A2'...","[1234.56, 888.88]",stem


In [10]:
#write upper one into function
def findpossibleepitope(epitopes, species): 
    foundepitope = []
    index = 1  #need to adjust to correct indexed dataframe
    epitopelist = epitopes
    for i in epitopelist["enzyme list and presence 0,1 or ?"]:
        elist =[] #re-create to iterate through all rows and keep intact structure
        for key, value in i.items():
            if value == 1:
                elist.append(key)
        s = list(set(elist) & set(species))
        if s != []:
            foundepitope.append(epitopelist.loc[index])
            #should be appending the whole row
        else:
            print(f"There is no enzyme for this epitope {epitopelist.loc[index][0]}")
        index +=1

    foundepitopes = pd.DataFrame(foundepitope)
    return foundepitopes

findpossibleepitope(c, zfdemo)

There is no enzyme for this epitope Bisecting NG core
There is no enzyme for this epitope First and elongation of Galactose at NG 
There is no enzyme for this epitope N-glycan core Fucosylation
There is no enzyme for this epitope Type 1 LacNAc
There is no enzyme for this epitope Type 2 LacNAc


,Name of glycotope feature,used units,"enzyme list and presence 0,1 or ?",ion list,groups
1,NG core,"{'H': 3, 'N': 2}","{'MGAT1': 1, 'MGAT2': 1, 'MAN2A1': 1, 'MAN2A2'...","[1234.56, 888.88]",stem


In [11]:
print("this cells includes demo epitopes that don't check enzyme and peaklist")
#assigning core structure priority: largest core -> smallest core -> stem
#if core assignment successful -> try to extend based on possible "ext" epitopes
#if there's sugar unit left when no ext epitopes is possible, try to add "terminal" epitopes and ends of all values at 0
#if the enumeration cannot reach zero, then it means this ISNOT a valid structure
#DO NOT break the enumeration loop since we may get multiple assignment possible

#this step is HIGHLY POTENTIAL to get optimized by any of AI method since they do this kind of task better than human

#core structure
#arm number in the last
Bisecting_NG_core= ["Bisecting NG core", {"H":3, "N":3}, {"Bisecting enzyme": 1}, [], "core", 2]
Biante_NG_core= ["Bi-antennary N-glycan core", {"H":5, "N":4}, {}, [], "core", 2]
Triante_NG_core= ["Tri-antennary N-glycan core", {"H":6, "N":5}, {}, [], "core", 3]
Tetraante_NG_core = ["Tetra-antennary N-glycan core", {"H":7, "N":6}, {}, [], "core", 4]
Hybrid_NG_core = ["Bi-antennary hybrid N-glycan core", {"H":6, "N":3}, {}, [], "core", 1]

#stem structure, not included in the first demo
NG_core= ["NG core", {"H":3, "N":2}, {}, [], "stem"]
NG_elongationGal = ["First and elongation of Galactose at NG ", {"H":1}, {}, [], "ext"] #re?

#higher priority to add core fucosylation if the enzyme is there
corefuc = ["N-glycan core Fucosylation", {"F":1}, {"FUT8":1}, [452.2490, 697.3753], "ter, coreter"]


#calculate larger terminal structure to use out terminal units, from largest one
#maybe we can apply a sort method to do this based on sum of all key values in sugar unit dict

#calculate start from "each arm?"
#another idea here: try to put as much big units as possible for all arms
#for example, 3 arms, S x 3, F x 3, H x 5, N x 5 is left after core substraction

#example2: 2 arms, S x 2, F x 3, H x 2, N x 2

#calculate average S and F in each arm first
# ex1: S = 1, F = 1 while H, N = 1.66
# ex2: S = 1, F = 1.5 while H, N = 1
# when one value is larger than 1, try to assign a larger one epitope first.

#here we consider all same "composition" as one structure to validate our guess.
#in next exp we'll add enzyme list and peak list to compare and make sure which defines it's real possible structure... 


#if ratio N>H, add LDNA if enzyme permits it to do so
LacDiNAc = ["LacDiNAc", {"N":2}, {}, [505], "ext, ter"]

#for zebrafish case, we have certain structure need to be consider first
zf_specific_epitope_5Ac = ["a1-4 Galx2 + GlcNAc + a2-3Sia + a1-3Fuc", {"F":1, "H":2, "N":1, "S":1}, {"unknown": "?"}, [1203], "ext"]
zf_specific_epitope_5Gc = ["a1-4 Galx2 + GlcNAc + a2-3Sia-Gc + a1-3Fuc", {"F":1, "H":2, "N":1, "G":1}, {"unknown": "?"}, [1233], "ext"]


#2 Fucose
LeBY = ["Lewis BY antigen", {"F":2, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ter, type1"]#b3-Gal type1 w a4-Fuc, a2-Fuc at Gal

#S+Fucose
sLeAX = ["Sialyl-Lewis AX antigen", {"F":1, "H":1, "N":1, "S":1}, {"unknown enzyme": "?"}, [0.0], "ext, ter, type1"]

#1 Fucose
Htype = ["Htype1 2 LNAc", {"F":1, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ter, type1"]
LeAX = ["Lewis AX antigen", {"F":1, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ext, type1"]#b3-Gal type1 w a4-Fuc

#1 Sialic acid
Sia_LNAc = ["Sialylated LNAc", {"H":1, "N":1, "S":1}, {}, [376, 580, 792, 825], "ter, type1"]
SiaG_LNAc = ["SialylatedGc LNAc", {"H":1, "N":1, "G":1}, {}, [406, 610, 822, 855], "ter, type1"]
KDN_LNAc = ["SialylatedKDN LNAc", {"H":1, "N":1, "KDN":1}, {}, [406, 610, 822, 855], "ter, type1"]


#zf_GalExtension = ["a1-4 Galx2 + GlcNAc", {"H":2, "N":1}, {"unknown": "?"}, [668], "ext"]
#Sia6_LacDiNAc = ["a2-6Sia_LacDiNAc", {"N":2}, {}, [505], "ext, ter"]


#general extension if there is more extending units available
LacNAc = ["LacNAc", {"H":1, "N":1}, {"B3GALT2":1, "B3GALT5":1}, [464], "ext, type1"]

#orphan units adds in the end
Neu5Ac = ["Neu5Ac sialic acid", {"S": 1}, {"CMAS": 1}, [376],  "ext, ter"]
Neu5Gc = ["Neu5Gc sialic acid", {"G": 1}, {"CMAH": 1}, [406], "ext, ter"]
#KDN = ["KDN sialic acid", {"KDN": 1}, {"CMP-KDN synthetase": 1}, [335], "ter"]
alphaGal = ["a1-4 Galx2 + GlcNAc orphan count", {"H":1}, {"unknown": "?"}, [668], "ter"]
#5Ac and 5Gc can be added on any arms with Sia at terminal

demo_epitopes = [
    Bisecting_NG_core, 
    Biante_NG_core,
    Triante_NG_core,
    Tetraante_NG_core,
    Hybrid_NG_core,
    corefuc,
    LeBY,
    sLeAX,
    Htype,
    LeAX,
    Sia_LNAc,
    SiaG_LNAc,
    KDN_LNAc,
    LacNAc,
    Neu5Ac,
    Neu5Gc,
    alphaGal,
]
print(demo_epitopes)

this cells includes demo epitopes that don't check enzyme and peaklist
[['Bisecting NG core', {'H': 3, 'N': 3}, {'Bisecting enzyme': 1}, [], 'core', 2], ['Bi-antennary N-glycan core', {'H': 5, 'N': 4}, {}, [], 'core', 2], ['Tri-antennary N-glycan core', {'H': 6, 'N': 5}, {}, [], 'core', 3], ['Tetra-antennary N-glycan core', {'H': 7, 'N': 6}, {}, [], 'core', 4], ['Bi-antennary hybrid N-glycan core', {'H': 6, 'N': 3}, {}, [], 'core', 1], ['N-glycan core Fucosylation', {'F': 1}, {'FUT8': 1}, [452.249, 697.3753], 'ter, coreter'], ['Lewis BY antigen', {'F': 2, 'H': 1, 'N': 1}, {'unknown enzyme': '?'}, [0.0], 'ter, type1'], ['Sialyl-Lewis AX antigen', {'F': 1, 'H': 1, 'N': 1, 'S': 1}, {'unknown enzyme': '?'}, [0.0], 'ext, ter, type1'], ['Htype1 2 LNAc', {'F': 1, 'H': 1, 'N': 1}, {'unknown enzyme': '?'}, [0.0], 'ter, type1'], ['Lewis AX antigen', {'F': 1, 'H': 1, 'N': 1}, {'unknown enzyme': '?'}, [0.0], 'ext, type1'], ['Sialylated LNAc', {'H': 1, 'N': 1, 'S': 1}, {}, [376, 580, 792, 825], 'te

In [12]:
import re

#finding and defining epitopes into groups
#should convert them into classes, methods to simplify this part of code...
print("testing cores")
coreelements = []
for i in demo_epitopes:
    if ("core" in i):
        coreelements.append(i)
        print(i)
        
extelements = []
print("testing extentives")
for i in demo_epitopes:
    if re.search("ext", i[4]):
        extelements.append(i)
        print(i)
        
        
terelements = []
print("testing terminal elements")
for i in demo_epitopes:
    if re.search("ter", i[4]):
        terelements.append(i)
        print(i)

#print(f"Core elements are {coreelements}")
#print(f"Extensive elements are {extelements}")
#print(f"Terminal elements are {terelements}")
print(f"compare same class w/ larger unit count should be iterated first.")
iter1 = []
orig = 0
for rank in coreelements:
    r = sum(rank[1].values())
    iter1.append((orig, r))
    orig+=1
    #print(iter1)
#sorting https://stackoverflow.com/questions/3121979/how-to-sort-a-list-tuple-of-lists-tuples-by-the-element-at-a-given-index
sorted_by_second = sorted(iter1, key=lambda tup: tup[1], reverse=True)
print(sorted_by_second)
sorted_coreelements = []
print("sorted list")
for sss in sorted_by_second:
    #print(sss[0])   #the original position
    sorted_coreelements.append(coreelements[sss[0]])
    print(coreelements[sss[0]])


iter2 = []
orig1 = 0
for rank in terelements:
    r = sum(rank[1].values())
    iter2.append((orig1, r))
    orig1+=1
    #print(f"{rank} iter2")
#sorting https://stackoverflow.com/questions/3121979/how-to-sort-a-list-tuple-of-lists-tuples-by-the-element-at-a-given-index
sorted_by_second1 = sorted(iter2, key=lambda tup: tup[1], reverse=True)
print(sorted_by_second1)
sorted_terelements = []
print("sorted terminal list")
for sss in sorted_by_second1:
    #print(sss[0])   #the original position
    sorted_terelements.append(terelements[sss[0]])
    print(terelements[sss[0]])

    
iter3 = []
orig2 = 0
for rank in extelements:
    r = sum(rank[1].values())
    iter3.append((orig2, r))
    orig2+=1
    #print(f"{rank} iter2")
#sorting https://stackoverflow.com/questions/3121979/how-to-sort-a-list-tuple-of-lists-tuples-by-the-element-at-a-given-index
sorted_by_second2 = sorted(iter3, key=lambda tup: tup[1], reverse=True)
print(sorted_by_second2)
sorted_extelements = []
print("sorted extensive list")
for sss in sorted_by_second2:
    #print(sss[0])   #the original position
    sorted_extelements.append(extelements[sss[0]])
    print(extelements[sss[0]])
    

testing cores
['Bisecting NG core', {'H': 3, 'N': 3}, {'Bisecting enzyme': 1}, [], 'core', 2]
['Bi-antennary N-glycan core', {'H': 5, 'N': 4}, {}, [], 'core', 2]
['Tri-antennary N-glycan core', {'H': 6, 'N': 5}, {}, [], 'core', 3]
['Tetra-antennary N-glycan core', {'H': 7, 'N': 6}, {}, [], 'core', 4]
['Bi-antennary hybrid N-glycan core', {'H': 6, 'N': 3}, {}, [], 'core', 1]
testing extentives
['Sialyl-Lewis AX antigen', {'F': 1, 'H': 1, 'N': 1, 'S': 1}, {'unknown enzyme': '?'}, [0.0], 'ext, ter, type1']
['Lewis AX antigen', {'F': 1, 'H': 1, 'N': 1}, {'unknown enzyme': '?'}, [0.0], 'ext, type1']
['LacNAc', {'H': 1, 'N': 1}, {'B3GALT2': 1, 'B3GALT5': 1}, [464], 'ext, type1']
['Neu5Ac sialic acid', {'S': 1}, {'CMAS': 1}, [376], 'ext, ter']
['Neu5Gc sialic acid', {'G': 1}, {'CMAH': 1}, [406], 'ext, ter']
testing terminal elements
['N-glycan core Fucosylation', {'F': 1}, {'FUT8': 1}, [452.249, 697.3753], 'ter, coreter']
['Lewis BY antigen', {'F': 2, 'H': 1, 'N': 1}, {'unknown enzyme': '?'},

In [26]:
import ast

def predcomplsttodict(complist):
    predcomp = {}
    if len(complist) >= 5:
        predcomp["F"] = complist[0]
        predcomp["H"] = complist[1]
        predcomp["N"] = complist[2]
        predcomp["S"] = complist[3]
        predcomp["G"] = complist[4]
        if len(complist) == 6:
            predcomp["KDN"] = complist[5]
    print(f"predicted composition dict is {predcomp}")
    return(predcomp)

def checkifpossiblededuce(invalflag): #for debugging
    if (invalflag is True):
        print("This structural enumeration is not possible")
    elif(invalflag is False):
        print("Applying calculations...")
    else:
        raise ValueError("you haven't check if the enumeration is possible")
        
def compactcompcalc(calccomp, enumerateepitope): #separate the elumarating METHOD outside as a independent function
        invalflag = None #init
        remaining_unit = [] #flag
        remaining_log = [] #Logs
        print(f"iteration on {enumerateepitope[0]}")
        valdict = {units: calccomp[units] - enumerateepitope[1].get(units, 0) for units in calccomp}
        print(f"current valdict in compactcompcalc {valdict}")
        for unittest in valdict.values():
            if unittest < 0:
                invalflag = True
                #print("caught negative value")
                remaining_unit.append(False)
            else:
                remaining_unit.append(True)
                pass
        if (False in remaining_unit) is False:  #why this statement works?
            invalflag = False
        checkifpossiblededuce(invalflag)
        if invalflag is False:
            print("add to logs")
            remaining_log.append(enumerateepitope[0]) #need to combine
            return True, remaining_log, valdict #added valdict
        elif invalflag is True:
            print("do nothing. Test next glycotope")
            return False, "", valdict
        else:
            raise ValueError("The core assignment is somhow not executed.")

def testzero(dictinput):
    for unittest in dictinput.values():
        if (unittest != 0): #set it invalid and leave no logs recorded
            return False
        else: #is it ok?
            print("All values are zero")
            return True

            
def decuder(specific_epitopes, spectrapredcomp):
    print("Try to assign NG core decreasing")
    calc_comp = spectrapredcomp #store to another dict to avoid updating on original data needed later
    #print(calc_comp) #already dict with sugar units only
    
    #define groups for the nested iteration later
    group_NGcore = sorted_coreelements  #first level
    group_terminal = sorted_terelements #second level
    group_extensive = sorted_extelements #final level
    #how to calculate sugar unit changes: #cdict = {key: calc[key] - bdict.get(key, 0) for key in adict}
    #logs were init here but maybe it's a mistake
    print("checking core elements")
    for corededuce in group_NGcore:
        logs = [] #enable logs for tracing back after interpretation
        invalflag = None #init
        reminflag = [] #init
        remaining_unit = []
        print(f"iteration on {corededuce[0]}") #shows which core we're on
        valdict = {units: calc_comp[units] - corededuce[1].get(units, 0) for units in calc_comp} #try to apply core
        print(valdict)
        for unittest in valdict.values():
            if unittest < 0: #set it invalid and leave no logs recorded
                invalflag = True
                #print("caught negative value")
                remaining_unit.append(False)
            else: #is it ok?
                remaining_unit.append(True)
                #pass
        #need to set flags valid outside the FOR iteration on calculated glycan units (we only assign it once after seeing all entries)
        if (False in remaining_unit) is False:  #why this statement works?
            invalflag = False
        checkifpossiblededuce(invalflag)
        if invalflag is False:
            print("Continue on extensivble epitope enumeration based on selected core type")
            logs.append(corededuce[0])
            #try to add glycotopes from as much arms as possible. when not possible, -1 arm
            armsno = corededuce[-1]
            while armsno > 0:
                print(f"Current arms number is {armsno}")
                print("Calculate average elements carrying on arms")
                avgunit = {}
                for reminame, reminno in valdict.items():
                    avgunit[reminame] = (reminno/armsno)
                print(avgunit)
                print("If none of units showing more than one, add all of them to first arm")
                for name, reminno in avgunit.items():  #use average unit to get correct calc answer
                    if reminno < 1:
                        #I think adding 1 flag and put the terminal-extensive loop back one level is correct
                        reminflag.append(False)
                        #print(f"there is no integer unit left :{name}, {reminno}")
                    elif reminno >= 1.0:
                        print(f"integer unit left :{name}, {reminno}, able for adding terminal structure first")
                        print(f"current units left: {valdict}, try to add units")
                        reminflag.append(True)
                        #calculate terminal
                        #print(f"it's on {name} now")
                if (True in reminflag): #moved one level back...seems it runs twice
                    for i in group_terminal:
                        #print(f"testing on {i[0]}")
                        a, b, secvaldict = compactcompcalc(valdict, i)
                        #a = compactcompcalc(valdict, i)[0]
                        #b = compactcompcalc(valdict, i)[1]
                        #secvaldict =  compactcompcalc(valdict, i)[2]
                        #print(f"a= {a}, b={b}, typeof = {type(b)}") #debug
                        c = "".join(b)
                        #print(f"c ={c}, type of c = {type(c)}") #debug
                        if a: #need to add logs if successfully predict
                            print("Iterate on extensive units")
                            logs.append(c)
                            print(f"testing: logs are {logs} right now")
                            #the flag isn't working...
                            for j in group_extensive:
                                aa, bb, finvaldict = compactcompcalc(secvaldict, j)
                                cc = "".join(bb)
                                print(f"need to check if final valdict is set to zero {finvaldict} on {j[0]}")
                                if aa:
                                    print("Add extensive glycotopes...")
                                    logs.append(cc)
                                    print("Finished testing on extensive units")
                                    finalresult = testzero(finvaldict)
                                    print(f"if final composition becomes zero: {finalresult}")
                                    if finalresult:
                                        print(f"return logs: {logs}")
                                        print("Should be saved somewhere")
                                    else:
                                        print("No logs possible")
                                else:
                                    print("The extensive units aren't possible to be added")
                                #print(f"testing {j}")
                                #compactcompcalc(valdict, j) #need to add logs if successfully predict
                        else:
                            print("Skipped iterating on extensive units after adding terminal")
                    print(f"current logs after core-terminal-extensive nested loop \n{logs}")
                    print(f"Suspended now. Start adding extensive ones w/o terminal units")    
                    #for k in group_extensive:
                    #    compactcompcalc(valdict, k) #need to add logs if successfully predict
                else:
                    print("reminflag is False, should be no further calculation")
                
                print(f"arms=arms-1")
                armsno-=1
                #first know if there is bisecting glycan w/ one orphan N
                #for avgepitope in terminalepitopes:
                    #try to generally assign same epitopes onto all arms if possible <- does this make sense?
                    #while (True in remaining_unit):
                        #do examine check #compactcompcalc(calccomp, enumerateepitope) to dete)ct terminal epitopes 
                #when to stop and start checking any if extending units?
                #when no more F S G KDN possible for assigning terminal epitopes -> fill elongation units
                    #another for loop to "deal with" units left that can only being partially added
                    #re-calulate the average units for (arms-1) until arms=1  <- kind of iterate?

                #if none of these core are possible, calc the "stem" instead (and we had already rejected possibilities of known core types.)
                #sort by total unit numbers and iterate through all possibilities 
                    #3rd layer for loop  compactcompcalc(calccomp, enumerateepitope) to fill the remaining units
                
        elif invalflag is True:
            print("do nothing. Test next core type")
            pass
        else:
            raise ValueError("The core assignment is somhow not executed.")
    #start recompose the structure
    print(f"logs are {logs}")
    #for bisecting ng this raise error after chaning it into not bisecting core... need to consider how to formulate it. Savepoint
    '''
    if (logs is []) or ( (Bisecting_NG_core == logs[0]) and valdict["N"] == 1) : #later one is for temp solution when not binding search w/ glycoTs and ions
        print("Need to calculate if there is only one N left")
        orphanN = {"N":1}
        valdict2 = {key: valdict[key] - orphanN.get(key, 0) for key in valdict}
        rema = False
        for i in valdict2.values():
            #detect any non-zero values
            if i > 0:
                print("valdict has xxx over 0")
                rema = True
        print(valdict2)
        if rema is False:
            print("It's Bi-antennary core w/o Gal")
        elif ((rema is True) and valdict2["F"] == 1):
            print("It's Bi-antennary core w/o Gal but has core-Fuc")
            logs.append("Bi-antennary stem")
    print("end of enumeration")
    return(logs[-1])
    '''
    
def deducecompositiona(specific_epitopes, speccompositions):
    print("developing, now testing without enzyme and fragments")
    #create index first
    index = 0
    #print(f"len of input compositions: {len(speccompositions)}")  #this calculated all strs so it will give like 6~10 len of each comp
    print("Input pd.Series")
    strlist1 = speccompositions["predictedcomp"]
    list2 = ast.literal_eval(strlist1)
    for j in range(len(list2)):
        #print(type(list2[j][1]))
        #print(list2[j][1])
        complist = ast.literal_eval(list2[j][1]) #convert string to tuple
        spectrapredcomp = predcomplsttodict(complist) #convert tuple to dict (add KDN if structure has "6" components)
        #this extracted tuple value is what we're going to calculate...or manipulate? in real.
        #decuder(specific_epitopes, spectrapredcomp)
        decuder(demo_epitopes, spectrapredcomp)

#calculate time spend
start = time.time()
#read 
sample1=pd.read_csv('Annotated_revised_zfNGintenstine_20230612_cloud.csv', sep='\t')

deduceinput = findpossibleepitope(c, zfdemo)
speccompositions = sample1.head(1) #24 for one spectrum has multiple assignments
tmpseries = sample1.iloc[23]
print(type(sample1.iloc[2])) #series
strlist123 = tmpseries["predictedcomp"]
print(f"the strlist is {strlist123}")

#deducecompositiona(deduceinput, speccompositions)  #for df
deducecompositiona(deduceinput, tmpseries)
#print(type(speccompositions))  #it's pandas df
sssss = time.time()-start
middle = time.time()
print('running through the load csv to strucutre prediction', sssss, 'seconds.')
#https://stackoverflow.com/questions/36459969/how-to-convert-a-list-to-a-dictionary-with-indexes-as-values isn't working

#second prompt
tmpseries1 = sample1.iloc[2]
deducecompositiona(deduceinput, tmpseries1)
print('running through the load csv to strucutre prediction', time.time()-start, 'seconds.')
print('delta time from previous analysis is',  time.time()-middle, 'seconds more.')


There is no enzyme for this epitope Bisecting NG core
There is no enzyme for this epitope First and elongation of Galactose at NG 
There is no enzyme for this epitope N-glycan core Fucosylation
There is no enzyme for this epitope Type 1 LacNAc
There is no enzyme for this epitope Type 2 LacNAc
<class 'pandas.core.series.Series'>
the strlist is [(1843.954, '(0, 4, 4, 0, 0, 0)'), (1845.945, '(0, 3, 2, 1, 0, 1)')]
developing, now testing without enzyme and fragments
Input pd.Series
predicted composition dict is {'F': 0, 'H': 4, 'N': 4, 'S': 0, 'G': 0, 'KDN': 0}
Try to assign NG core decreasing
checking core elements
iteration on Tetra-antennary N-glycan core
{'F': 0, 'H': -3, 'N': -2, 'S': 0, 'G': 0, 'KDN': 0}
This structural enumeration is not possible
do nothing. Test next core type
iteration on Tri-antennary N-glycan core
{'F': 0, 'H': -2, 'N': -1, 'S': 0, 'G': 0, 'KDN': 0}
This structural enumeration is not possible
do nothing. Test next core type
iteration on Bi-antennary N-glycan cor

In [14]:
#calculations only works on dict, not list or tuple... meaning we worked in vain on making them list
#https://stackoverflow.com/questions/17671875/how-to-subtract-values-from-dictionaries
adict = {"F":1, "H":5 , "N": 4, "S":1}
bdict = {"H":3, "N":2}
cdict = {key: adict[key] - bdict.get(key, 0) for key in adict}
print(cdict)

{'F': 1, 'H': 2, 'N': 2, 'S': 1}


In [37]:
art = tmpseries1["predictedcomp"]
print(art)
print(tmpseries1)
#artificail input
artseries1 = pd.Series(["[(2048.053, '(0, 5, 4, 0, 0, 0)')]", 1], index=["predictedcomp", "MS1scan no"]) #bianten core test
print(artseries1["predictedcomp"])
strlist1 = artseries1["predictedcomp"]
list2 = ast.literal_eval(strlist1)
print(list2)

[(1639.854, '(0, 3, 4, 0, 0, 0)')]
MS1scan no                                                                 12144
MS1Isolation mass                                                     820.432129
MS1monoIsomass                                                        820.432129
chargeState                                                                    2
man_check                                                                     ok
Theoretical mass                                                       1639.8543
in [H+]                                                              1639.856418
Structure                                                                   H3N4
Structure dict                 {'F': 0, 'H': '3', 'N': '4', 'S': 0, 'G': 0, '...
MS2 Scan no                                                                12180
peaklist                       MS2peaklist(dMass=(90.99137115478516, 94.06515...
predictedcomp                                 [(1639.854, '(0, 3, 4, 0, 0,

In [38]:
deducecompositiona(deduceinput, artseries1)

developing, now testing without enzyme and fragments
Input pd.Series
predicted composition dict is {'F': 0, 'H': 5, 'N': 4, 'S': 0, 'G': 0, 'KDN': 0}
Try to assign NG core decreasing
checking core elements
iteration on Tetra-antennary N-glycan core
{'F': 0, 'H': -2, 'N': -2, 'S': 0, 'G': 0, 'KDN': 0}
This structural enumeration is not possible
do nothing. Test next core type
iteration on Tri-antennary N-glycan core
{'F': 0, 'H': -1, 'N': -1, 'S': 0, 'G': 0, 'KDN': 0}
This structural enumeration is not possible
do nothing. Test next core type
iteration on Bi-antennary N-glycan core
{'F': 0, 'H': 0, 'N': 0, 'S': 0, 'G': 0, 'KDN': 0}
Applying calculations...
Continue on extensivble epitope enumeration based on selected core type
Current arms number is 2
Calculate average elements carrying on arms
{'F': 0.0, 'H': 0.0, 'N': 0.0, 'S': 0.0, 'G': 0.0, 'KDN': 0.0}
If none of units showing more than one, add all of them to first arm
reminflag is False, should be no further calculation
arms=arms-